# Notebook 5 — Learned Decoding: a Graph Neural Network Decoder

*Part of **QEC Explorer**, and the bridge to the research this project connects to. In Notebooks 2 and 3 we watched **belief propagation fail** on the surface code — it has no threshold here. The frontier response is to **learn** a decoder from data. This notebook builds a small but real **graph neural network (GNN)** decoder in PyTorch, trains it on synthetic noise, and compares it to the BP baseline — honestly.*

This is the notebook where we connect to current research: **AlphaQubit** (Google, 2024) and a wave of neural decoders are exactly this idea at scale. Ours is a teaching-sized version — and, importantly, **we report the real result even though it's a negative one at this scale.** That honesty is the point: you'll see *why* learned decoders are hard, not a rigged demo.

> **What's genuinely real here:** the GNN trains (you'll see the loss curve fall), and we measure its logical error rate against BP on the *same* test errors. **What we're honest about:** at $d=3$ with a couple minutes of CPU training, the GNN does **not** beat BP. We explain why, and what a real (much larger) model does differently.

> 📦 Needs PyTorch — preinstalled in Colab. (`pip install torch` locally if needed.)

---
## 1 · Reuse the verified physics + BP baseline (verbatim from Notebook 2)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

np.random.seed(0)
plt.rcParams["figure.dpi"] = 110

# ---- from Notebook 1: data qubits, stabilizers, syndrome (ports of lattice-core.js) ----
def build_data_qubits(d):
    return [(r, c) for r in range(d) for c in range(d)]

def build_stabilizers(d):
    stabs = []
    for R in range(-1, d):
        for C in range(-1, d):
            stype = "Z" if (R + C) % 2 == 0 else "X"
            corners = [(R, C), (R, C + 1), (R + 1, C), (R + 1, C + 1)]
            cand = [(rr, cc) for (rr, cc) in corners if 0 <= rr < d and 0 <= cc < d]
            if len(cand) == 0:
                continue
            on_top_bottom = (R == -1 or R == d - 1)
            on_left_right = (C == -1 or C == d - 1)
            if len(cand) == 2:
                if stype == "Z" and not on_left_right: continue
                if stype == "X" and not on_top_bottom: continue
            if len(cand) not in (2, 4):
                continue
            stabs.append({"type": stype, "center": (C + 0.5, R + 0.5), "data": cand})
    return stabs

def compute_syndrome(stabs, errors):
    syn = []
    for s in stabs:
        parity = 0
        for (r, c) in s["data"]:
            e = errors.get((r, c))
            if not e: continue
            if s["type"] == "Z" and e.get("x"): parity ^= 1
            if s["type"] == "X" and e.get("z"): parity ^= 1
        syn.append(parity)
    return syn

d = 3
data = build_data_qubits(d)
stabs = build_stabilizers(d)
print(f"d={d}: {len(data)} data qubits, {len(stabs)} stabilizers — code ready to decode.")

In [ ]:
def combine_errors(a, b):
    # XOR two error sets per Pauli component. A qubit ending in identity is dropped.
    out = {}
    for k in set(a) | set(b):
        ea = a.get(k, {"x": False, "z": False})
        eb = b.get(k, {"x": False, "z": False})
        x = bool(ea.get("x")) ^ bool(eb.get("x"))
        z = bool(ea.get("z")) ^ bool(eb.get("z"))
        if x or z:
            out[k] = {"x": x, "z": z}
    return out

def error_weight(errors):
    # number of non-identity qubits (Y counts as 1 — it's one physical qubit)
    return sum(1 for k in errors if errors[k].get("x") or errors[k].get("z"))

def diff_keys(a, b):
    # qubits where two corrections differ in either component (for degeneracy checks)
    out = []
    for k in set(a) | set(b):
        ea = a.get(k, {"x": False, "z": False}); eb = b.get(k, {"x": False, "z": False})
        if bool(ea.get("x")) != bool(eb.get("x")) or bool(ea.get("z")) != bool(eb.get("z")):
            out.append(k)
    return sorted(out)

# quick sanity: applying an error to itself cancels to identity
e = {(1,1): {"x": True, "z": False}}
print("error XOR itself =", combine_errors(e, e), " (empty = cancelled ✓)")

In [ ]:
# We need logical_status from Notebook 1 to judge residuals. (Port of logicalStatus.)
def logical_status(stabs, errors, d):
    syn = compute_syndrome(stabs, errors)
    if any(syn):
        return {"logical": False, "detectable": True}
    x_col0 = z_row0 = 0
    for r in range(d):
        e = errors.get((r, 0))
        if e and e.get("x"): x_col0 ^= 1
    for c in range(d):
        e = errors.get((0, c))
        if e and e.get("z"): z_row0 ^= 1
    return {"logical": (x_col0 == 1 or z_row0 == 1), "detectable": False}

def evaluate_correction(stabs, original_errors, correction, d):
    # Port of evaluateCorrection(). Returns status in {fixed, logical-introduced, left-codespace}.
    residual = combine_errors(original_errors, correction)
    if any(compute_syndrome(stabs, residual)):
        return {"status": "left-codespace", "ok": False, "label": "Did not return to codespace ✗"}
    if logical_status(stabs, residual, d)["logical"]:
        return {"status": "logical-introduced", "ok": False, "label": "Logical error introduced ✗"}
    return {"status": "fixed", "ok": True, "label": "Fixed ✓"}

# sanity: the perfect correction (== the error) always fixes a detectable error
err = {(1,1): {"x": True, "z": False}}
print("perfect correction verdict:", evaluate_correction(stabs, err, err, d)["label"])

In [ ]:
DETECTOR_TYPE = {"X": "Z", "Z": "X"}   # channel -> the stabilizer type that detects it

def stabs_of_type(stabs, stype):
    # return [(index, stabilizer), ...] for stabilizers of one type
    return [(i, s) for i, s in enumerate(stabs) if s["type"] == stype]

def apply_flips(correction, qubit_keys, channel):
    # XOR a channel-flip onto each listed qubit (mutates `correction`)
    for k in qubit_keys:
        cur = correction.get(k, {"x": False, "z": False}).copy()
        if channel == "X": cur["x"] = not cur["x"]
        else:              cur["z"] = not cur["z"]
        if not cur["x"] and not cur["z"]: correction.pop(k, None)
        else:                              correction[k] = cur

print("X channel is read by", DETECTOR_TYPE["X"], "-type stabilizers;",
      "Z channel by", DETECTOR_TYPE["Z"], "-type.")

In [ ]:
BP_CHANNEL_P   = 0.05
BP_MAX_ITERS   = 20
BP_CONVERGE_EPS = 1e-3

def _clamp(x): return max(-30.0, min(30.0, x))

def bp_channel(stabs, data, errors, channel, trace=False):
    # Port of bpChannel(): LLR sum-product message passing on the Tanner graph.
    det_type = DETECTOR_TYPE[channel]
    checks = stabs_of_type(stabs, det_type)              # [(idx, stab)]
    syn = compute_syndrome(stabs, errors)

    var_keys = list(data)
    var_index = {k: i for i, k in enumerate(var_keys)}
    check_vars = [[var_index[k] for k in s["data"]] for (_, s) in checks]
    check_syn  = [syn[i] for (i, _) in checks]
    var_checks = [[] for _ in var_keys]
    for ci, vs in enumerate(check_vars):
        for vi in vs: var_checks[vi].append(ci)

    L0 = np.log((1 - BP_CHANNEL_P) / BP_CHANNEL_P)       # prior LLR (favors "no error")
    Lvc = [{vi: L0 for vi in vs} for vs in check_vars]   # var->check messages
    Lcv = [{vi: 0.0 for vi in vs} for vs in check_vars]  # check->var messages

    iters, converged, iter_marg = 0, False, []
    for it in range(BP_MAX_ITERS):
        iters = it + 1
        max_change = 0.0
        # check -> variable (tanh / sum-product rule, syndrome bit as sign)
        for ci, vs in enumerate(check_vars):
            sign = -1.0 if check_syn[ci] else 1.0
            for vi in vs:
                prod = 1.0
                for vj in vs:
                    if vj == vi: continue
                    prod *= np.tanh(_clamp(Lvc[ci][vj]) / 2.0)
                prod = max(-0.999999, min(0.999999, prod))
                msg = sign * 2.0 * np.arctanh(prod)
                max_change = max(max_change, abs(msg - Lcv[ci][vi]))
                Lcv[ci][vi] = msg
        # variable -> check
        for vi in range(len(var_keys)):
            cs = var_checks[vi]
            total = sum(Lcv[ci][vi] for ci in cs)
            for ci in cs:
                Lvc[ci][vi] = _clamp(L0 + (total - Lcv[ci][vi]))
        # per-iteration marginals  P(error) = sigmoid(-Lmarg)
        marg = [1.0 / (1.0 + np.exp(_clamp(L0 + sum(Lcv[ci][vi] for ci in var_checks[vi]))))
                for vi in range(len(var_keys))]
        if trace: iter_marg.append(marg)
        if max_change < BP_CONVERGE_EPS:
            converged = True; break

    correction = {}
    final_marg = [1.0 / (1.0 + np.exp(_clamp(L0 + sum(Lcv[ci][vi] for ci in var_checks[vi]))))
                  for vi in range(len(var_keys))]
    for vi, k in enumerate(var_keys):
        if final_marg[vi] > 0.5: apply_flips(correction, [k], channel)
    return {"correction": correction, "iters": iters, "converged": converged,
            "final_marg": final_marg, "iter_marg": iter_marg, "var_keys": var_keys}

def bp_decode(stabs, data, errors, trace=False):
    # Port of bpDecode(): both channels, merge.
    x = bp_channel(stabs, data, errors, "X", trace)
    z = bp_channel(stabs, data, errors, "Z", trace)
    iters = max(x["iters"], z["iters"])
    converged = x["converged"] and z["converged"]
    note = (f"Converged after {iters} iteration{'s' if iters != 1 else ''}."
            if converged else f"Did not converge after {BP_MAX_ITERS} iterations.")
    return {"correction": combine_errors(x["correction"], z["correction"]),
            "iters": iters, "converged": converged, "note": note, "channels": {"X": x, "Z": z}}

res = bp_decode(stabs, data, {(1,1): {"x": True, "z": False}}, trace=True)
print("BP on single X(1,1):", res["note"])
print("correction:", res["correction"], " ->",
      evaluate_correction(stabs, {(1,1): {"x": True}}, res["correction"], d)["label"])

In [ ]:
import numpy as np, torch, torch.nn as nn, time
torch.manual_seed(0); np.random.seed(0)
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"]=110

d=3
data=build_data_qubits(d); stabs=build_stabilizers(d)
# evaluate_correction + bp_decode came from Notebook 2 cells above
assert evaluate_correction(stabs,{(1,1):{"x":True}},
        bp_decode(stabs,data,{(1,1):{"x":True}})["correction"],d)["ok"]
print("✅ Notebook 2 physics + BP loaded and working (BP fixes single X(1,1)).")

## 2 · The noise sampler (from Notebook 3)
We train on synthetic data: random errors sampled exactly as in the Monte Carlo notebook.

In [ ]:
def sample_error(data,p,model="depolarizing",bias=1,rng=None):
    rng=rng or np.random
    pz=min(1.0,p*bias) if model=="biased" else p
    e={}
    for (r,c) in data:
        x=rng.random()<p; z=rng.random()<pz
        if x or z: e[(r,c)]={"x":bool(x),"z":bool(z)}
    return e
print("noise sampler ready.")

---
## 3 · The GNN decoder

The decoder is a **message-passing network on the Tanner graph**:
- **Data-qubit nodes** (9 of them at $d=3$) — what we want to predict a flip for.
- **Stabilizer/check nodes** (8 of them) — each carries its **syndrome bit** (lit or not) and its **type** (X or Z).
- **Edges** connect each stabilizer to the data qubits it touches (the same adjacency the stabilizers already define).

Messages pass **stab → data** and **data → stab** for several rounds (a GRU cell updates each node's hidden state), then a readout head predicts, for every data qubit, the probability it carries an **X** flip and a **Z** flip. This is structurally analogous to BP — but the update rules are **learned**, not the fixed `tanh` LLR rule.

In [ ]:
# ---- build the Tanner-graph adjacency as fixed tensors ----
ND=len(data); NS=len(stabs)
data_idx={k:i for i,k in enumerate(data)}
edges=[(data_idx[k],si) for si,s in enumerate(stabs) for k in s["data"]]
E=np.array(edges)
ed_d=torch.tensor(E[:,0],dtype=torch.long)   # data endpoint of each edge
ed_s=torch.tensor(E[:,1],dtype=torch.long)   # stab endpoint of each edge
stab_type_t=torch.tensor([[1.0 if s["type"]=="Z" else 0.0] for s in stabs])
print(f"Tanner graph: {ND} data nodes, {NS} stab nodes, {len(edges)} edges.")

H=32
class GNNDecoder(nn.Module):
    def __init__(self, rounds=4):
        super().__init__()
        self.d_in=nn.Linear(1,H); self.s_in=nn.Linear(2,H)
        self.msg_sd=nn.Linear(H,H); self.msg_ds=nn.Linear(H,H)
        self.upd_d=nn.GRUCell(H,H); self.upd_s=nn.GRUCell(H,H)
        self.readout=nn.Sequential(nn.Linear(H,H),nn.ReLU(),nn.Linear(H,2))
        self.rounds=rounds
    def forward(self, syndrome):           # syndrome: (B, NS)
        B=syndrome.shape[0]
        hd=self.d_in(torch.zeros(B,ND,1)).reshape(B,ND,H)
        sfeat=torch.cat([syndrome.unsqueeze(2),
                         stab_type_t.unsqueeze(0).expand(B,NS,1)],dim=2)
        hs=self.s_in(sfeat)
        for _ in range(self.rounds):
            m=self.msg_sd(hs)
            agg=torch.zeros(B,ND,H); agg.index_add_(1,ed_d,m[:,ed_s,:])
            hd=self.upd_d(agg.reshape(-1,H),hd.reshape(-1,H)).reshape(B,ND,H)
            m=self.msg_ds(hd)
            agg=torch.zeros(B,NS,H); agg.index_add_(1,ed_s,m[:,ed_d,:])
            hs=self.upd_s(agg.reshape(-1,H),hs.reshape(-1,H)).reshape(B,NS,H)
        return self.readout(hd)            # (B, ND, 2) logits: [P(X), P(Z)]

model=GNNDecoder(rounds=4)
print(f"GNN params: {sum(p.numel() for p in model.parameters())}")

### A real pitfall, stated up front
Most qubits are **not** in error (at $p=0.08$, only ~8% of labels are positive). A naive model can score high "accuracy" by always predicting *no flip* — and decode nothing. We fight this two ways: a **positive-class weight** in the loss, and we judge the model by **logical error rate** (does the correction actually work?), never by raw per-bit accuracy.

In [ ]:
def make_dataset(n,p,model_name,bias,rng):
    S=np.zeros((n,NS),dtype=np.float32); Y=np.zeros((n,ND,2),dtype=np.float32); errs=[]
    for i in range(n):
        e=sample_error(data,p,model_name,bias,rng); errs.append(e)
        S[i]=np.array(compute_syndrome(stabs,e),dtype=np.float32)
        for k,ev in e.items():
            j=data_idx[k]
            Y[i,j,0]=1.0 if ev["x"] else 0.0
            Y[i,j,1]=1.0 if ev["z"] else 0.0
    return torch.tensor(S),torch.tensor(Y),errs

P_TRAIN=0.08
rng=np.random.default_rng(7)
Str,Ytr,_       = make_dataset(4000,P_TRAIN,"depolarizing",1,rng)
Sva,Yva,errs_va = make_dataset(1500,P_TRAIN,"depolarizing",1,rng)
pos=Ytr.mean().item(); pw=torch.tensor((1-pos)/max(pos,1e-3))
print(f"label positive rate {pos:.3f} -> pos_weight {pw.item():.2f}")

---
## 4 · Train it (real training, on CPU)

In [ ]:
opt=torch.optim.Adam(model.parameters(),lr=3e-3)
lossfn=nn.BCEWithLogitsLoss(pos_weight=pw)
EPOCHS=30; BS=256; losses=[]
t0=time.time()
for ep in range(EPOCHS):
    model.train(); perm=torch.randperm(Str.shape[0]); tot=0; nb=0
    for i in range(0,Str.shape[0],BS):
        idx=perm[i:i+BS]
        loss=lossfn(model(Str[idx]),Ytr[idx])
        opt.zero_grad(); loss.backward(); opt.step()
        tot+=loss.item(); nb+=1
    losses.append(tot/nb)
    if ep%5==0 or ep==EPOCHS-1: print(f"epoch {ep:2d}  loss {tot/nb:.4f}")
print(f"trained in {time.time()-t0:.1f}s on CPU")

In [ ]:
plt.figure(figsize=(6,3.6))
plt.plot(losses,color="#9b6dff",lw=2)
plt.xlabel("epoch"); plt.ylabel("training loss (BCE)")
plt.title("GNN decoder training — the loss really does fall")
plt.grid(True,ls=":",color="0.85"); plt.tight_layout(); plt.show()
assert losses[-1] < losses[0]*0.8, "loss did not fall meaningfully — training failed"
print(f"✅ Loss fell from {losses[0]:.3f} to {losses[-1]:.3f} — the model learned something.")

---
## 5 · The honest comparison: GNN vs BP on the *same* test errors

We decode every test error two ways — with the trained GNN and with BP — and measure each one's **logical error rate** with the *same* `evaluate_correction` used everywhere in this project. Identical test set, fair fight.

In [ ]:
def gnn_correction(logits_i):
    probs=torch.sigmoid(logits_i)
    corr={}
    for j,k in enumerate(data):
        x=probs[j,0].item()>0.5; z=probs[j,1].item()>0.5
        if x or z: corr[k]={"x":x,"z":z}
    return corr

model.eval()
with torch.no_grad(): logits=model(Sva)
gnn_fail=bp_fail=0
for i,e in enumerate(errs_va):
    if not evaluate_correction(stabs,e,gnn_correction(logits[i]),d)["ok"]: gnn_fail+=1
    if not evaluate_correction(stabs,e,bp_decode(stabs,data,e)["correction"],d)["ok"]: bp_fail+=1
N=len(errs_va)
gnn_ler=gnn_fail/N; bp_ler=bp_fail/N
print(f"=== {N} identical test errors, d={d}, p={P_TRAIN}, depolarizing ===")
print(f"  GNN logical error rate : {gnn_ler:.4f}")
print(f"  BP  logical error rate : {bp_ler:.4f}")
print(f"  -> GNN {'BEATS' if gnn_ler<bp_ler else 'does NOT beat'} BP at this scale.")

In [ ]:
plt.figure(figsize=(4.6,3.8))
bars=plt.bar(["GNN\n(learned)","BP\n(baseline)"],[gnn_ler,bp_ler],
             color=["#9b6dff","#2ec4a0"],width=0.6)
plt.ylabel("logical error rate (lower = better)")
plt.title(f"Honest result at d={d}, p={P_TRAIN}")
for b,v in zip(bars,[gnn_ler,bp_ler]):
    plt.text(b.get_x()+b.get_width()/2,v+0.01,f"{v:.3f}",ha="center",fontsize=10)
plt.ylim(0,max(gnn_ler,bp_ler)*1.25); plt.tight_layout(); plt.show()

---
## 6 · Why the GNN loses here — and why that's not the end of the story

Our small GNN trains cleanly but **doesn't beat BP at $d=3$**, and that's the honest, expected outcome. The reasons are real and worth understanding:

1. **No information BP doesn't already have.** Both decoders see only the syndrome. The fundamental **degeneracy** (Notebook 2's Z(1,2) trap) limits *every* syndrome-only decoder — a learned model can't conjure information that isn't there.
2. **Tiny code, tiny model, tiny training.** $d=3$ has 9 qubits and a handful of syndromes; there's little structure for a GNN to exploit that BP doesn't capture, and a few thousand samples over a few minutes is nothing compared to real training runs.
3. **Per-qubit independent readout.** Our head predicts each qubit's flip independently; it can't easily represent "*these qubits flip together or not at all*," which is exactly the correlated structure that matters near the boundary.

**What real learned decoders do differently.** Google's **AlphaQubit** (2024) uses a recurrent transformer, consumes **soft** (analog) syndrome information across **many** measurement rounds, trains on enormous datasets (and real device data), and *does* beat MWPM/BP on large codes. The idea in this notebook is the same; the scale is what makes it work. Our negative result is a faithful small-scale snapshot of a method that wins when you give it room.

> This is the honest bridge to the research connection: learned decoding is promising **and** hard, and the gap between a teaching demo and a state-of-the-art result is real engineering, not hand-waving.

---
## 7 · Wrap-up

You built a real GNN decoder — message passing on the Tanner graph, trained on synthetic noise — and measured it fairly against BP. It learned (the loss fell), but at this scale it didn't beat the baseline, and you now know exactly why.

- **→ Notebook 4 — qLDPC & bivariate-bicycle codes** — the other half of the frontier: better *codes*, not just better decoders.
- **→ [QEC Explorer · research connection](research.html)** — how this project's decoder and code threads connect to current work (and where the arXiv manuscript will be linked).
- **→ [Module 3 · Noise Explorer (live)](https://github.com/kondshk/QEC-Explorer)** — the Monte Carlo this notebook's evaluation is built on.

The decoders, the threshold, and now a learned decoder — you've walked the whole path from a single qubit flip to the research frontier.